<a href="https://colab.research.google.com/github/AiyazKhan-123/Emotional-Triggers-and-Social-Network-Structures-in-Misinformation-Propagation/blob/main/Misinformation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip -q install pandas numpy scikit-learn statsmodels transformers torch
!git clone --depth 1 https://github.com/KaiDMML/FakeNewsNet.git


Cloning into 'FakeNewsNet'...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 26 (delta 1), reused 14 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (26/26), 18.63 MiB | 20.49 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("/content/FakeNewsNet/dataset")

files = {
    "politifact_fake.csv": "politifact",
    "gossipcop_fake.csv": "gossipcop",
}

frames = []

for filename, source in files.items():
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Required file missing: {path}")

    df = pd.read_csv(path, keep_default_na=False)
    df.columns = df.columns.str.strip().str.lower()

    # The dataset uses 'news_url' instead of 'url'
    required = {"id", "news_url", "title", "tweet_ids"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{filename} is missing columns: {missing}")

    df = df.rename(columns={"news_url": "url"})
    df["source"] = source
    df["article_id"] = source + ":" + df["id"].astype(str)
    df["label"] = 1
    frames.append(df)

articles = pd.concat(frames, ignore_index=True)
articles["title"] = articles["title"].astype(str).str.strip()
articles = articles[articles["title"].ne("")].copy()
articles = articles.drop_duplicates(subset=["article_id"])

print("Fake articles loaded:", len(articles))
print(articles.groupby("source").size())
display(articles.head())

Fake articles loaded: 5755
source
gossipcop     5323
politifact     432
dtype: int64


,id,url,title,tweet_ids,source,article_id,label
0,politifact15014,speedtalk.com/forum/viewtopic.php?t=51650,BREAKING: First NFL Team Declares Bankruptcy O...,937349434668498944\t937379378006282240\t937380...,politifact,politifact:politifact15014,1
1,politifact15156,politics2020.info/index.php/2018/03/13/court-o...,Court Orders Obama To Pay $400 Million In Rest...,972666281441878016\t972678396575559680\t972827...,politifact,politifact:politifact15156,1
2,politifact14745,www.nscdscamps.org/blog/category/parenting/467...,UPDATE: Second Roy Moore Accuser Works For Mic...,929405740732870656\t929439450400264192\t929439...,politifact,politifact:politifact14745,1
3,politifact14355,https://howafrica.com/oscar-pistorius-attempts...,Oscar Pistorius Attempts To Commit Suicide,886941526458347521\t887011300278194176\t887023...,politifact,politifact:politifact14355,1
4,politifact15371,http://washingtonsources.org/trump-votes-for-d...,Trump Votes For Death Penalty For Being Gay,915205698212040704\t915242076681506816\t915249...,politifact,politifact:politifact15371,1


In [ ]:
import torch
from transformers import pipeline
import pandas as pd
from pathlib import Path

device = 0 if torch.cuda.is_available() else -1

emotion_model = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None,
    device=device
)

# Start with a pilot. Change to all articles after validation.
PILOT_MODE = True
sample_size = min(500, len(articles))

work = articles.head(sample_size).copy() if PILOT_MODE else articles.copy()

scores = []
batch_size = 16

# Define the expected emotion labels in lowercase to avoid missing columns
expected_emotions = {"anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"}

for start in range(0, len(work), batch_size):
    batch = work["title"].iloc[start:start + batch_size].tolist()
    outputs = emotion_model(batch, truncation=True)

    for result in outputs:
        # Pre-initialize all expected emotions with 0.0 to prevent KeyError
        score_dict = {emotion: 0.0 for emotion in expected_emotions}
        for item in result:
            label_lower = item["label"].lower()
            score_dict[label_lower] = item["score"]
        scores.append(score_dict)

emotion_df = pd.DataFrame(scores).fillna(0)
emotion_df.index = work.index

work = pd.concat(
    [work, emotion_df.add_prefix("emotion_")],
    axis=1
)

# Validate neutral score exists
if "neutral" not in emotion_df.columns:
    raise ValueError(
        "Model has no neutral score. Define and validate another "
        "emotion-intensity measure before proceeding."
    )

work["emotion_intensity"] = 1 - work["emotion_neutral"]

OUTPUT_DIR = Path("/content/FakeNewsNet_RQ1")
OUTPUT_DIR.mkdir(exist_ok=True)

work.to_csv(
    OUTPUT_DIR / "fake_articles_with_emotions.csv",
    index=False
)

print("Saved:", len(work), "articles")
print("Pilot mode:", PILOT_MODE)
display(work[[
    "article_id", "source", "title", "emotion_intensity"
]].head())

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Saved: 500 articles
Pilot mode: True


,article_id,source,title,emotion_intensity
0,politifact:politifact15014,politifact,BREAKING: First NFL Team Declares Bankruptcy O...,0.836177
1,politifact:politifact15156,politifact,Court Orders Obama To Pay $400 Million In Rest...,0.435481
2,politifact:politifact14745,politifact,UPDATE: Second Roy Moore Accuser Works For Mic...,0.120392
3,politifact:politifact14355,politifact,Oscar Pistorius Attempts To Commit Suicide,0.981615
4,politifact:politifact15371,politifact,Trump Votes For Death Penalty For Being Gay,0.883180


In [ ]:

required_columns = [
    "article_id",
    "cascade_size",
    "emotion_intensity",
    "emotion_anger",
    "emotion_fear",
    "seed_centrality",
    "network_clustering",
    "network_modularity",
    "collection_coverage",
]

# This is a schema template, NOT measured research data.
schema_template = pd.DataFrame(columns=required_columns)
schema_template.to_csv(
    OUTPUT_DIR / "cascade_schema_template.csv",
    index=False
)

print(schema_template.columns.tolist())


['article_id', 'cascade_size', 'emotion_intensity', 'emotion_anger', 'emotion_fear', 'seed_centrality', 'network_clustering', 'network_modularity', 'collection_coverage']


In [ ]:

import statsmodels.formula.api as smf
from pathlib import Path

CASCADE_PATH = OUTPUT_DIR / "cascade_features.csv"

if not CASCADE_PATH.exists():
    raise FileNotFoundError(
        "H1-H4 cannot yet be tested. First create cascade_features.csv "
        "from observed social posts and network data. The starter CSV "
        "does not contain cascade sizes or network measurements."
    )

df = pd.read_csv(CASCADE_PATH)

required = {
    "article_id", "cascade_size", "emotion_intensity",
    "seed_centrality", "network_clustering",
    "network_modularity", "collection_coverage"
}

missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

# Check one observation per article
if df["article_id"].duplicated().any():
    raise ValueError(
        "Expected one row per article/cascade. Resolve duplicates first."
    )

# Outcome must be a non-negative count
df["cascade_size"] = pd.to_numeric(
    df["cascade_size"], errors="coerce"
)

if (df["cascade_size"].dropna() < 0).any():
    raise ValueError("cascade_size cannot be negative.")

if not np.allclose(
    df["cascade_size"].dropna() % 1, 0
):
    raise ValueError("cascade_size must contain integer counts.")

# Remove missing required fields; never convert them to zero
model_cols = list(required - {"article_id"})
df = df.dropna(subset=model_cols).copy()

if len(df) < 30:
    print("WARNING: Small sample. Estimates may be unstable.")

# Standardize continuous predictors for comparable coefficient scales.
from sklearn.preprocessing import StandardScaler

predictors = [
    "emotion_intensity", "seed_centrality",
    "network_clustering", "network_modularity",
    "collection_coverage"
]

scaler = StandardScaler()
df[[c + "_z" for c in predictors]] = scaler.fit_transform(
    df[predictors].astype(float)
)

# H1: Emotion and cascade size
h1 = smf.negativebinomial(
    "cascade_size ~ emotion_intensity_z + collection_coverage_z",
    data=df
).fit(disp=False, maxiter=300)

print("\nH1 — Emotion")
print(h1.summary())

# H2: Seed-user centrality and cascade size
h2 = smf.negativebinomial(
    "cascade_size ~ seed_centrality_z + collection_coverage_z",
    data=df
).fit(disp=False, maxiter=300)

print("\nH2 — Network centrality")
print(h2.summary())

# H3: Network structure and cascade size
h3 = smf.negativebinomial(
    "cascade_size ~ network_clustering_z + "
    "network_modularity_z + collection_coverage_z",
    data=df
).fit(disp=False, maxiter=300)

print("\nH3 — Network structure")
print(h3.summary())

# Exponentiated coefficients are incidence-rate ratios.
for name, model in [("H1", h1), ("H2", h2), ("H3", h3)]:
    result = pd.DataFrame({
        "coefficient": model.params,
        "IRR": np.exp(model.params),
        "CI_lower_IRR": np.exp(model.conf_int()[0]),
        "CI_upper_IRR": np.exp(model.conf_int()[1]),
        "p_value": model.pvalues,
    })
    print(f"\n{name} — IRR and confidence intervals")
    display(result)



H1 — Emotion
                     NegativeBinomial Regression Results                      
Dep. Variable:           cascade_size   No. Observations:                  500
Model:               NegativeBinomial   Df Residuals:                      497
Method:                           MLE   Df Model:                            2
Date:                Thu, 15 Oct 2026   Pseudo R-squ.:                0.006075
Time:                        18:48:02   Log-Likelihood:                -3092.4
converged:                       True   LL-Null:                       -3111.3
Covariance Type:            nonrobust   LLR p-value:                 6.188e-09
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 5.7486      0.076     75.993      0.000       5.600       5.897
emotion_intensity_z      -0.3327      0.078     -4.270      0.000      -0.485      -

,coefficient,IRR,CI_lower_IRR,CI_upper_IRR,p_value
Intercept,5.748596,313.749911,270.516435,363.892887,0.000000e+00
emotion_intensity_z,-0.332668,0.717008,0.615481,0.835282,1.951007e-05
collection_coverage_z,-0.209153,0.811271,0.701840,0.937764,4.667285e-03
alpha,2.857558,17.418947,13.070891,23.213391,1.033899e-84



H2 — IRR and confidence intervals


,coefficient,IRR,CI_lower_IRR,CI_upper_IRR,p_value
Intercept,5.762198,318.046714,274.095596,369.045376,0.000000e+00
seed_centrality_z,0.293558,1.341191,1.154811,1.557653,1.203272e-04
collection_coverage_z,-0.223235,0.799927,0.692457,0.924077,2.424337e-03
alpha,2.875390,17.732335,13.284886,23.668679,7.938624e-85



H3 — IRR and confidence intervals


,coefficient,IRR,CI_lower_IRR,CI_upper_IRR,p_value
Intercept,5.803554,331.475368,285.273286,385.160213,0.000000e+00
network_clustering_z,0.009626,1.009673,0.873554,1.167002,8.963411e-01
network_modularity_z,0.018528,1.018701,0.858486,1.208816,8.319303e-01
collection_coverage_z,-0.300587,0.740384,0.644247,0.850866,2.277514e-05
alpha,2.929455,18.717423,13.955330,25.104523,3.639093e-85


In [ ]:

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_absolute_error, mean_poisson_deviance

# Common control features are included in every model.
emotion_features = ["emotion_intensity_z"]
network_features = [
    "seed_centrality_z",
    "network_clustering_z",
    "network_modularity_z",
]
common_numeric = ["collection_coverage_z"]

# Add source as a common control if present.
categorical_features = ["source"] if "source" in df.columns else []

# One split for every model; keep related claims together when possible.
if "claim_group_id" in df.columns:
    groups = df["claim_group_id"].fillna(
        df["article_id"]
    ).astype(str)

    splitter = GroupShuffleSplit(
        n_splits=1, test_size=0.25, random_state=42
    )
    train_idx, test_idx = next(
        splitter.split(df, df["cascade_size"], groups)
    )
    train, test = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()
else:
    train, test = train_test_split(
        df, test_size=0.25, random_state=42
    )

y_train = train["cascade_size"].astype(float)
y_test = test["cascade_size"].astype(float)

feature_sets = {
    "Emotion only": emotion_features + common_numeric,
    "Network only": network_features + common_numeric,
    "Combined": emotion_features + network_features + common_numeric,
}

results = []

for model_name, numeric_cols in feature_sets.items():
    transformers = [
        ("numeric", make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler()
        ), numeric_cols)
    ]

    if categorical_features:
        transformers.append((
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ))

    preprocess = ColumnTransformer(transformers)

    model = make_pipeline(
        preprocess,
        PoissonRegressor(alpha=1.0, max_iter=1000)
    )

    model.fit(train, y_train)
    predictions = np.maximum(model.predict(test), 1e-9)

    results.append({
        "model": model_name,
        "MAE": mean_absolute_error(y_test, predictions),
        "Poisson_deviance": mean_poisson_deviance(
            y_test, predictions
        ),
    })

comparison = pd.DataFrame(results).sort_values("MAE")
display(comparison)

print(
    "H4: the combined model is better only if its held-out "
    "performance improves over BOTH individual feature models. "
    "Use repeated splits or bootstrap intervals to assess uncertainty."
)


,model,MAE,Poisson_deviance
0,Emotion only,388.640594,692.735146
2,Combined,399.893358,749.746293
1,Network only,432.884492,808.919131


H4: the combined model is better only if its held-out performance improves over BOTH individual feature models. Use repeated splits or bootstrap intervals to assess uncertainty.
